[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/08_deep_retrieval_ann/08_proyecto_two_tower_faiss.ipynb)

# Módulo 08 · Proyecto: retrieval two-tower + índice FAISS sirviendo top-K en milisegundos
### Proyecto — Curso «Sistemas de Recomendación: de cero a nivel Netflix»

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración estimada** | 4–6 h |
| **GPU recomendada** | L4 / A100 (T4 vale para ML-1M) |
| **Unidades de Colab estimadas** | ~4 (ML-1M) · ~20 (ML-25M) |
| **Prerrequisitos** | Lección 08 (y 01, 02, 05) |

> En Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU*. Todo el notebook
> funciona también en CPU con `FAST_DEV_RUN = True` (más lento, mismos conceptos).

## 🏢 Contexto de negocio
El ranker de CineMatch (proyectos 06–07) es bueno, pero solo ordena lo que le llega: hoy los candidatos salen de "populares + item-kNN" y el equipo sospecha que **faltan películas de nicho** relevantes. Te encargan la nueva **fuente de candidatos principal**: un two-tower que, dada la historia del usuario, devuelva **300 candidatos en < 10 ms p99 en CPU**.

## 📦 Dataset
MovieLens-1M (o **ML-25M** con `SCALE="25m"` en A100) con split temporal global (utilidades de módulos 01/02). Se evalúa sobre el **catálogo completo** filtrando vistos.

## ✅ Entregables y rúbrica
| # | Entregable | Criterio |
|---|---|---|
| 1 | `TwoTower` (torre usuario con historia + demografía; torre ítem con id + géneros + año) | Tests de forma/normalización pasan |
| 2 | Pérdida in-batch con **logQ** y máscara de *accidental hits* | Test incluido pasa |
| 3 | Recall@100 en test | > baseline de popularidad (en split temporal la popularidad es un rival fuerte; con ML-1M real y `FAST_DEV_RUN=False` apunta a ≥ 1,2×) |
| 4 | Ablación: sin logQ / con logQ / + MNS | Tabla + análisis cabeza/cola |
| 5 | Índice FAISS (HNSW o IVF) | Recall@100 ≥ 0,95 respecto a búsqueda exacta (catálogo real, consultas = usuarios reales) |
| 6 | Servicio `recommend(user)` con filtrado de vistos | p99 < 10 ms en CPU (1 hilo, máquina sin otras cargas) para 300 candidatos |

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import random, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)
if device == "cuda":
    print(torch.cuda.get_device_name(0))
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

### 🧰 Utilidades mínimas (vienen de los módulos 01 y 02)

Para que este notebook sea **autocontenido en Colab** repetimos aquí, en versión
compacta, tres piezas que ya construiste antes:

1. `load_movielens(size)` — descarga MovieLens desde GroupLens (módulo 01). Si no hay
   red, genera un **MovieLens sintético** con el mismo esquema para que nada se bloquee.
2. `temporal_split(...)` — split **temporal global** (módulo 01): entrenamos con el
   pasado y evaluamos con el futuro, como en producción.
3. `recall_at_k`, `ndcg_at_k` — métricas de ranking de top-K (módulo 02).

In [ ]:
# === Utilidades mínimas (módulos 01 y 02) ===================================
import os, io, zipfile, urllib.request
import numpy as np
import pandas as pd

ML_URLS = {
    "1m": "https://files.grouplens.org/datasets/movielens/ml-1m.zip",
    "25m": "https://files.grouplens.org/datasets/movielens/ml-25m.zip",
}

def _synthetic_movielens(n_users=1500, n_items=900, seed=42):
    """MovieLens 'de juguete' con el mismo esquema que ML-1M (fallback sin red)."""
    rng = np.random.default_rng(seed)
    genres = ["Action", "Comedy", "Drama", "Thriller", "Romance", "Sci-Fi",
              "Horror", "Animation", "Documentary", "Children's"]
    G = len(genres)
    item_g = rng.dirichlet(np.ones(G) * 0.3, n_items)            # mezcla de géneros
    user_g = rng.dirichlet(np.ones(G) * 0.5, n_users)            # gustos
    pop = np.minimum(rng.zipf(1.4, n_items), 300).astype(float)  # cola larga
    activity = np.clip(rng.zipf(1.6, n_users) * 20, 20, 600).astype(int)
    rows, t0 = [], 956_703_932                                    # ~abril 2000
    for u in range(n_users):
        aff = item_g @ user_g[u]
        p = pop * (aff + 0.02) ** 2
        p /= p.sum()
        n = min(activity[u], n_items // 2)
        items = rng.choice(n_items, size=n, replace=False, p=p)
        score = 4 * (aff[items] / aff.max()) + rng.normal(0, 0.8, n)
        r = np.clip(np.round(1 + score), 1, 5).astype(int)
        start = t0 + rng.integers(0, 2 * 365 * 86400)
        ts = np.sort(start + rng.integers(0, 365 * 86400, n))
        rows.append(pd.DataFrame({"user_id": u + 1, "item_id": items + 1,
                                  "rating": r, "timestamp": ts}))
    ratings = pd.concat(rows, ignore_index=True)
    gnames = ["|".join([genres[g] for g in np.where(item_g[i] > 0.25)[0]] or
                       [genres[int(item_g[i].argmax())]]) for i in range(n_items)]
    years = rng.integers(1930, 2001, n_items)
    movies = pd.DataFrame({"item_id": np.arange(1, n_items + 1),
                           "title": [f"Película sintética {i+1} ({y})" for i, y in enumerate(years)],
                           "genres": gnames})
    users = pd.DataFrame({"user_id": np.arange(1, n_users + 1),
                          "gender": rng.choice(["M", "F"], n_users),
                          "age": rng.choice([1, 18, 25, 35, 45, 50, 56], n_users),
                          "occupation": rng.integers(0, 21, n_users),
                          "zip": [f"{z:05d}" for z in rng.integers(0, 99999, n_users)]})
    return ratings, movies, users

def load_movielens(size: str = "1m", data_dir: str = "data"):
    """Devuelve (ratings, movies, users) con columnas user_id, item_id, rating, timestamp."""
    os.makedirs(data_dir, exist_ok=True)
    try:
        zpath = os.path.join(data_dir, f"ml-{size}.zip")
        if not os.path.exists(zpath):
            print(f"Descargando MovieLens-{size} de GroupLens…")
            urllib.request.urlretrieve(ML_URLS[size], zpath)
        with zipfile.ZipFile(zpath) as z:
            if size == "1m":
                rd = lambda f, cols: pd.read_csv(io.BytesIO(z.read(f"ml-1m/{f}")), sep="::",
                                                 engine="python", names=cols, encoding="latin-1")
                ratings = rd("ratings.dat", ["user_id", "item_id", "rating", "timestamp"])
                movies = rd("movies.dat", ["item_id", "title", "genres"])
                users = rd("users.dat", ["user_id", "gender", "age", "occupation", "zip"])
            else:
                ratings = pd.read_csv(z.open("ml-25m/ratings.csv")).rename(
                    columns={"userId": "user_id", "movieId": "item_id"})
                movies = pd.read_csv(z.open("ml-25m/movies.csv")).rename(columns={"movieId": "item_id"})
                users = None
        print(f"MovieLens-{size} real: {len(ratings):,} ratings")
    except Exception as e:  # sin red / URL caída → fallback sintético
        print(f"⚠️ No se pudo descargar MovieLens ({type(e).__name__}). Uso datos SINTÉTICOS.")
        ratings, movies, users = _synthetic_movielens()
    return ratings, movies, users

def temporal_split(df: pd.DataFrame, val_frac=0.1, test_frac=0.1, ts_col="timestamp"):
    """Split temporal GLOBAL: [pasado | validación | futuro] por cuantiles de tiempo."""
    t_val, t_test = df[ts_col].quantile([1 - val_frac - test_frac, 1 - test_frac])
    train = df[df[ts_col] < t_val]
    val = df[(df[ts_col] >= t_val) & (df[ts_col] < t_test)]
    test = df[df[ts_col] >= t_test]
    return train, val, test

def recall_at_k(topk: np.ndarray, truth: list, k: int) -> float:
    """topk: [U, >=k] ids recomendados; truth: lista de sets con los relevantes."""
    vals = [len(set(row[:k]) & t) / len(t) for row, t in zip(topk, truth) if len(t) > 0]
    return float(np.mean(vals)) if vals else 0.0

def ndcg_at_k(topk: np.ndarray, truth: list, k: int) -> float:
    """NDCG@k con relevancia binaria (módulo 02)."""
    disc = 1.0 / np.log2(np.arange(2, k + 2))
    vals = []
    for row, t in zip(topk, truth):
        if not t:
            continue
        gains = np.array([1.0 if i in t else 0.0 for i in row[:k]])
        idcg = disc[: min(len(t), k)].sum()
        vals.append((gains * disc[: len(gains)]).sum() / idcg)
    return float(np.mean(vals)) if vals else 0.0

In [ ]:
FAST_DEV_RUN = True
SCALE = "1m"
MAX_EXAMPLES = 300_000 if FAST_DEV_RUN else None
MAX_EVAL_USERS = 2000 if FAST_DEV_RUN else 20000
EPOCHS = 3 if FAST_DEV_RUN else 10
BATCH = 1024 if FAST_DEV_RUN else 8192

In [ ]:
# === Datos para retrieval: secuencias, ejemplos (historia → siguiente ítem) y conjuntos de evaluación
from numpy.lib.stride_tricks import sliding_window_view
ratings, movies, users = load_movielens(SCALE)
movies = movies[movies.item_id.isin(ratings.item_id.unique())].reset_index(drop=True)
imap = {v: i + 1 for i, v in enumerate(movies.item_id)}            # 0 = padding
ratings = ratings[ratings.item_id.isin(imap)].copy(); ratings["iid"] = ratings.item_id.map(imap)
N_ITEMS = len(movies)
GENRES = sorted({g for gs in movies.genres for g in gs.split("|")})
ITEM_G = np.zeros((N_ITEMS + 1, len(GENRES)), np.float32)
for j, g in enumerate(GENRES):
    ITEM_G[1:, j] = movies.genres.str.contains(g, regex=False).values
years = movies.title.str.extract(r"\((\d{4})\)\s*$")[0].astype(float).fillna(1990).values
ITEM_Y = np.zeros(N_ITEMS + 1, np.int64); ITEM_Y[1:] = np.clip((years - 1910) // 5, 0, 22).astype(int) + 1
TITLE = np.array(["<pad>"] + movies.title.tolist(), dtype=object)
MAIN_GENRE = np.array(["<pad>"] + movies.genres.str.split("|").str[0].tolist(), dtype=object)

# Features demográficas (solo existen en ML-1M)
n_users_max = int(ratings.user_id.max()) + 1
UCAT = np.zeros((n_users_max, 3), np.int64)
if users is not None:
    u = users.set_index("user_id")
    UCAT[u.index, 0] = (u.gender == "F").astype(int).values + 1
    UCAT[u.index, 1] = pd.factorize(u.age, sort=True)[0] + 1
    UCAT[u.index, 2] = u.occupation.astype(int).values + 1
UCAT_DIMS = tuple(int(UCAT[:, j].max()) + 1 for j in range(3))

train, val, test = temporal_split(ratings)
H = 50
def sequences(df):
    return df.sort_values(["user_id", "timestamp"]).groupby("user_id").iid.apply(np.array).to_dict()

def train_examples(seqs, max_examples=None, seed=42):
    Hs, T, U = [], [], []
    for u, s in seqs.items():
        if len(s) < 2: continue
        w = sliding_window_view(np.concatenate([np.zeros(H, s.dtype), s]), H)[1:len(s)]   # historia ANTES de cada evento
        Hs.append(w); T.append(s[1:]); U.append(np.full(len(s) - 1, u))
    Hs, T, U = np.concatenate(Hs), np.concatenate(T), np.concatenate(U)
    if max_examples and len(T) > max_examples:
        i = np.random.default_rng(seed).choice(len(T), max_examples, replace=False); Hs, T, U = Hs[i], T[i], U[i]
    return Hs.astype(np.int64), T.astype(np.int64), U.astype(np.int64)

def eval_set(hist_df, target_df, max_users=None, seed=42):
    seqs = sequences(hist_df); truth = target_df.groupby("user_id").iid.agg(set)
    us = [u for u in truth.index if u in seqs]
    if max_users and len(us) > max_users:
        us = list(np.random.default_rng(seed).choice(us, max_users, replace=False))
    hist, seen, tr, uu = [], [], [], []
    for u in us:
        s = seqs[u]; t = truth[u] - set(s)
        if not t: continue
        h = s[-H:]; hist.append(np.pad(h, (H - len(h), 0))); seen.append(np.unique(s)); tr.append(t); uu.append(u)
    return {"hist": np.array(hist, np.int64), "ucat": UCAT[np.array(uu)], "seen": seen, "truth": tr, "users": np.array(uu)}

train_seqs = sequences(train)
Htr, Ttr, Utr = train_examples(train_seqs, max_examples=MAX_EXAMPLES)
EV_val = eval_set(train, val, MAX_EVAL_USERS)
EV_test = eval_set(pd.concat([train, val]), test, MAX_EVAL_USERS)
item_count = np.bincount(train.iid.values, minlength=N_ITEMS + 1).astype(np.float64)
print(f"{N_ITEMS} ítems | {len(Ttr):,} ejemplos de train | usuarios eval: val={len(EV_val['users'])}, test={len(EV_test['users'])}")

In [ ]:
pop_rank = np.argsort(-item_count)
def popularity_topk(EV, k=100):
    out = []
    for seen in EV["seen"]:
        s = set(seen); out.append([i for i in pop_rank[:k + len(s) + 1] if i not in s and i != 0][:k])
    return np.array(out)
def evaluate(topk, EV):
    return {"Recall@10": recall_at_k(topk, EV["truth"], 10), "Recall@100": recall_at_k(topk, EV["truth"], 100),
            "NDCG@10": ndcg_at_k(topk, EV["truth"], 10)}
POP = evaluate(popularity_topk(EV_test), EV_test); print("Popularidad:", POP)

## Paso 1 — Las dos torres
Requisitos: salida L2-normalizada de dimensión `d`; la torre de usuario usa la **media enmascarada** (padding = 0) de los embeddings de la historia y las 3 features demográficas `UCAT`; la torre de ítem usa `ITEM_G` (multi-hot de géneros) y `ITEM_Y` (quinquenio).

In [ ]:
class TwoTower(nn.Module):
    def __init__(self, n_items, n_genres, n_years=24, ucat_dims=(3, 8, 22), d=64):
        super().__init__()
        # TODO
        raise NotImplementedError
    def item_vec(self, ids):
        raise NotImplementedError
    def user_vec(self, hist, ucat):
        raise NotImplementedError

def _test_towers():
    m = TwoTower(N_ITEMS, len(GENRES), ucat_dims=UCAT_DIMS)
    v = m.item_vec(torch.tensor([1, 2, 3])); u = m.user_vec(torch.as_tensor(Htr[:4]), torch.as_tensor(UCAT[Utr[:4]]))
    assert v.shape == (3, 64) and u.shape == (4, 64)
    assert torch.allclose(v.norm(dim=-1), torch.ones(3), atol=1e-4) and torch.allclose(u.norm(dim=-1), torch.ones(4), atol=1e-4)
    print("✅ torres OK")
try:
    _test_towers()
except NotImplementedError:
    print("⏳ Implementa TwoTower")

## Paso 2 — La pérdida
`retrieval_loss(u, v, pos, log_p, tau, logq)` recibe ya los vectores de usuario `u [B,d]` y de ítem positivo `v [B,d]`, los ids `pos [B]` y `log_p` (log-frecuencia de cada ítem). Debe:
1. calcular `logits = u @ v.T / tau`,
2. si `logq`, restar `log(B · p_j)` a cada **columna**,
3. enmascarar *accidental hits* (mismo id en otra fila),
4. devolver la cross-entropy con la diagonal como etiqueta.

In [ ]:
def retrieval_loss(u, v, pos, log_p, tau=0.05, logq=True):
    # TODO
    raise NotImplementedError

def _test_loss():
    torch.manual_seed(0)
    u = F.normalize(torch.randn(4, 8), dim=-1); pos = torch.tensor([5, 5, 7, 9]); v = u.clone()
    lp = torch.log(torch.full((10,), 0.1))
    l1 = retrieval_loss(u, v, pos, lp, tau=0.1, logq=False)
    # filas 0 y 1 tienen el mismo positivo: con la máscara, su pérdida no incluye al "gemelo"
    logits = u @ v.T / 0.1; logits[0, 1] = logits[1, 0] = -1e9
    assert torch.allclose(l1, F.cross_entropy(logits, torch.arange(4)), atol=1e-5)
    l2 = retrieval_loss(u, v, pos, lp, tau=0.1, logq=True)                 # p uniforme → logQ no cambia la CE
    assert torch.allclose(l1, l2, atol=1e-5)
    print("✅ pérdida OK")
try:
    _test_loss()
except NotImplementedError:
    print("⏳ Implementa retrieval_loss")

## Paso 3 — Entrenamiento, evaluación y ablación
Entrena 3 variantes (sin logQ, con logQ, con logQ + 512 negativos uniformes) y compara Recall@100 total y por grupos de popularidad (cabeza 20 % / cola 40 %).

In [ ]:
# TODO: bucle de entrenamiento + retrieve() + evaluate() + tabla de ablación

## Paso 4 — Índice FAISS y servicio
Construye un índice sobre los vectores de ítem (y, para estresarlo, sobre un corpus aumentado de ≥ 200 k vectores), mide recall@100 vs exacto y latencia p50/p99 de `recommend()` con 1 hilo.

In [ ]:
import faiss
# TODO

---
# ⛔ SPOILER — Solución de referencia

In [ ]:
# === Two-tower desde cero ====================================================
class TwoTower(nn.Module):
    def __init__(self, n_items, n_genres, n_years=24, ucat_dims=(3, 8, 22), d=64):
        super().__init__()
        self.item_id = nn.Embedding(n_items + 1, d, padding_idx=0)        # tabla COMPARTIDA (ítem e historia)
        nn.init.normal_(self.item_id.weight, std=0.05)
        self.genre = nn.Linear(n_genres, d, bias=False)
        self.year = nn.Embedding(n_years, d)
        self.item_mlp = nn.Sequential(nn.Linear(d, 256), nn.ReLU(), nn.Linear(256, d))
        self.ucat = nn.ModuleList([nn.Embedding(c, d) for c in ucat_dims])
        self.user_mlp = nn.Sequential(nn.Linear(2 * d, 256), nn.ReLU(), nn.Linear(256, d))
        self.register_buffer("G", torch.as_tensor(ITEM_G)); self.register_buffer("Y", torch.as_tensor(ITEM_Y))

    def item_vec(self, ids):
        x = self.item_id(ids) + self.genre(self.G[ids]) + self.year(self.Y[ids])
        return F.normalize(x + self.item_mlp(x), dim=-1)                  # residual + L2 → coseno

    def user_vec(self, hist, ucat):
        m = (hist > 0).float().unsqueeze(-1)
        h = (self.item_id(hist) * m).sum(1) / m.sum(1).clamp(min=1)       # media de la historia
        demo = sum(e(ucat[:, j]) for j, e in enumerate(self.ucat))
        x = torch.cat([h, demo], -1)
        return F.normalize(self.user_mlp(x), dim=-1)

@torch.no_grad()
def all_item_vecs(model, bs=8192):
    model.eval()
    ids = torch.arange(1, N_ITEMS + 1, device=device)
    V = torch.cat([model.item_vec(ids[i:i + bs]) for i in range(0, len(ids), bs)])
    return torch.cat([torch.zeros(1, V.size(1), device=device), V])     # fila 0 = padding

@torch.no_grad()
def retrieve(model, EV, k=100, V=None, bs=1024):
    V = all_item_vecs(model) if V is None else V
    out = []
    for i in range(0, len(EV["hist"]), bs):
        u = model.user_vec(torch.as_tensor(EV["hist"][i:i + bs], device=device), torch.as_tensor(EV["ucat"][i:i + bs], device=device))
        s = u @ V.T; s[:, 0] = -1e9
        for r, seen in enumerate(EV["seen"][i:i + bs]):
            s[r, torch.as_tensor(seen, device=device)] = -1e9                 # filtrar ya vistos
        out.append(s.topk(k, dim=1).indices.cpu().numpy())
    return np.concatenate(out)

def evaluate(topk, EV):
    return {"Recall@10": recall_at_k(topk, EV["truth"], 10), "Recall@50": recall_at_k(topk, EV["truth"], 50),
            "Recall@100": recall_at_k(topk, EV["truth"], 100), "NDCG@10": ndcg_at_k(topk, EV["truth"], 10)}

In [ ]:
_test_towers()
LOG_P = torch.as_tensor(np.log((item_count + 1) / (item_count + 1).sum()), dtype=torch.float32, device=device)

def retrieval_loss(u, v, pos, log_p, tau=0.05, logq=True):
    B = pos.size(0)
    logits = u @ v.T / tau
    if logq:
        logits = logits - (log_p[pos] + math.log(B))[None, :]
    accidental = (pos[None, :] == pos[:, None]) & ~torch.eye(B, dtype=torch.bool, device=pos.device)
    logits = logits.masked_fill(accidental, -1e9)
    return F.cross_entropy(logits, torch.arange(B, device=pos.device))

_test_loss()

def train_tt(logq=True, n_uniform=0, tau=0.05, epochs=EPOCHS, bs=BATCH):
    torch.manual_seed(seed)
    model = TwoTower(N_ITEMS, len(GENRES), ucat_dims=UCAT_DIMS).to(device); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    Hs, Ts, Uc = (torch.as_tensor(a, device=device) for a in (Htr, Ttr, UCAT[Utr]))
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Ts), device=device)
        for s0 in range(0, len(Ts), bs):
            i = perm[s0:s0 + bs]
            u, v = model.user_vec(Hs[i], Uc[i]), model.item_vec(Ts[i])
            loss = retrieval_loss(u, v, Ts[i], LOG_P, tau, logq)
            if n_uniform:                                                   # MNS: columnas extra uniformes
                neg = torch.randint(1, N_ITEMS + 1, (n_uniform,), device=device)
                ln = u @ model.item_vec(neg).T / tau - (math.log(n_uniform / N_ITEMS) if logq else 0.0)
                logits = u @ v.T / tau
                if logq: logits = logits - (LOG_P[Ts[i]] + math.log(len(i)))[None, :]
                acc = (Ts[i][None, :] == Ts[i][:, None]) & ~torch.eye(len(i), dtype=torch.bool, device=device)
                allp = torch.cat([logits.masked_fill(acc, -1e9), ln.masked_fill(neg[None, :] == Ts[i][:, None], -1e9)], 1)
                loss = F.cross_entropy(allp, torch.arange(len(i), device=device))
            opt.zero_grad(); loss.backward(); opt.step()
        print(f"  época {ep+1}: Recall@100 val = {evaluate(retrieve(model, EV_val), EV_val)['Recall@100']:.4f}")
    return model

abl, models = {"Popularidad": POP}, {}
for name, kw in {"sin logQ": dict(logq=False), "logQ": dict(logq=True), "logQ + MNS": dict(logq=True, n_uniform=512)}.items():
    print("▶", name); models[name] = train_tt(**kw); abl[name] = evaluate(retrieve(models[name], EV_test), EV_test)
pd.DataFrame(abl).T.round(4)

In [ ]:
pct = np.argsort(np.argsort(-item_count)) / N_ITEMS
grp = {"cabeza": pct < 0.2, "cola": pct >= 0.6}
rows = {}
for name, m in models.items():
    tk = retrieve(m, EV_test)
    rows[name] = {g: recall_at_k(tk, [{i for i in t if mask[i]} for t in EV_test["truth"]], 100) for g, mask in grp.items()}
display(pd.DataFrame(rows).T.round(4))
best = max(models, key=lambda n: abl[n]["Recall@100"]); model = models[best]
print(f"Mejor: {best} · Recall@100 = {abl[best]['Recall@100']:.4f} · ratio vs popularidad = {abl[best]['Recall@100']/POP['Recall@100']:.2f}×")

In [ ]:
faiss.omp_set_num_threads(1)
V = all_item_vecs(model).cpu().numpy()[1:].astype(np.float32)
d = V.shape[1]
# 1) Índice sobre el catálogo real
index = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT); index.hnsw.efConstruction = 100; index.add(V); index.hnsw.efSearch = 256
with torch.no_grad():
    Q = model.user_vec(torch.as_tensor(EV_test["hist"][:1000], device=device), torch.as_tensor(EV_test["ucat"][:1000], device=device)).cpu().numpy()
exact = faiss.IndexFlatIP(d); exact.add(V)
_, GT = exact.search(Q, 100); _, I = index.search(Q, 100)
print("Catálogo real · recall@100 HNSW vs exacto:", np.mean([len(set(a) & set(b)) / 100 for a, b in zip(I, GT)]).round(4))
# 2) Estrés: corpus aumentado
rng_ = np.random.default_rng(seed); NC = 200_000 if FAST_DEV_RUN else 2_000_000
i1, i2, w_ = rng_.integers(0, len(V), NC), rng_.integers(0, len(V), NC), rng_.uniform(0.5, 1.0, (NC, 1))
C = w_ * V[i1] + (1 - w_) * V[i2] + rng_.normal(0, 0.5 / np.sqrt(d), (NC, d)); C = (C / np.linalg.norm(C, axis=1, keepdims=True)).astype(np.float32)
big = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT); big.hnsw.efConstruction = 80
faiss.omp_set_num_threads(4); big.add(C); faiss.omp_set_num_threads(1)
ex_big = faiss.IndexFlatIP(d); ex_big.add(C); _, GTb = ex_big.search(Q[:300], 100)
for ef in [64, 128, 256, 512]:
    big.hnsw.efSearch = ef
    t0 = time.perf_counter(); _, Ib = big.search(Q[:300], 100); dt = (time.perf_counter() - t0) / 300 * 1e3
    print(f"Corpus {NC:,} · efSearch={ef}: recall@100={np.mean([len(set(a)&set(b))/100 for a,b in zip(Ib, GTb)]):.4f} · {dt:.2f} ms/consulta")

In [ ]:
def recommend(user_hist: np.ndarray, ucat: np.ndarray, k=300, over_fetch=1.5):
    with torch.no_grad():
        q = model.user_vec(torch.as_tensor(user_hist[None], device=device), torch.as_tensor(ucat[None], device=device)).cpu().numpy()
    seen = set(user_hist[user_hist > 0].tolist())
    _, I = index.search(q.astype(np.float32), int(k * over_fetch) + len(seen))
    return [int(i) + 1 for i in I[0] if i >= 0 and (i + 1) not in seen][:k]

index.hnsw.efSearch = 256
lat = []
for j in range(min(500, len(EV_test["hist"]))):
    t0 = time.perf_counter(); recommend(EV_test["hist"][j], EV_test["ucat"][j]); lat.append((time.perf_counter() - t0) * 1e3)
p50, p99 = np.percentile(lat, [50, 99])
print(f"recommend(): p50 = {p50:.2f} ms · p99 = {p99:.2f} ms  → {'✅' if p99 < 10 else '❌'} objetivo p99 < 10 ms")
plt.figure(figsize=(6, 3)); plt.hist(lat, bins=40); plt.axvline(p99, c="r", ls="--", label=f"p99={p99:.2f} ms"); plt.legend()
plt.xlabel("ms por petición"); plt.title("Latencia del servicio de candidatos"); plt.show()
print([TITLE[i] for i in recommend(EV_test["hist"][0], EV_test["ucat"][0], k=10)])

## 🚀 Retos extra
1. **ML-25M en A100** con `BATCH=16384`: ¿cuánto sube el recall al crecer el batch? (in-batch negatives "gratis").
2. **Hard negatives** (puestos 20–100 del modelo actual tras 1 época de warm-up) mezclados con MNS.
3. **Estimador de frecuencia en streaming** en lugar de la frecuencia exacta para logQ.
4. **Multi-interés**: varios vectores por usuario (clustering de su historia, PinnerSage) y unión de resultados.
5. **IVF-PQ + refine** a 2 M vectores: compara memoria y p99 con HNSW.
6. **Qdrant o pgvector** en Docker con filtro "solo títulos disponibles en el país del usuario".

## 🤔 Reflexión (producción / MLOps)
- ¿Cómo despliegas un nuevo modelo si la torre de usuario y el índice de ítems deben ser **de la misma versión**? (blue/green del índice, módulo 16-17).
- ¿Cada cuánto re-indexas? ¿Qué pasa con los estrenos que llegan entre reentrenos? (torre de ítem con features de contenido → embeddings para ítems nuevos sin reentrenar).
- ¿Qué métricas online monitorizarías del retrieval (recall respecto a lo que el ranker acaba eligiendo, diversidad de fuentes, latencia p99)?